# 04 · Regex baseline

Phase 5: a deliberately naive extractor for three entity types in Kenyan judgments:

| Entity | Examples |
|---|---|
| `DATE` | `13/7/2022`, `25th March 2026`, `March 25, 2026` |
| `MONEY` | `Kshs. 80,000/=`, `KES 1,500`, `Ksh.7,409.50` |
| `CASE_REF` | `Civil Appeal No. E031 of 2023`, `[2026] KEHC 3922 (KLR)`, `[2018] eKLR` |

**Naive** means: one simple regular expression per format seen in the EDA, no context, no machine learning, and no special cases added to fix problems found along the way. The code is in `src/baseline.py`. This notebook runs it on the cleaned slice and looks at the output. It is **not** scored here: scoring needs hand labels, which come in Phase 6.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from data import load_case_law, load_slice, clean_slice
from baseline import PATTERNS, extract

In [2]:
case_law = load_case_law()
df, _ = clean_slice(load_slice(case_law), case_law)
len(df)

381

## 1. The patterns

In [3]:
for entity, patterns in PATTERNS.items():
    for name, pattern in patterns.items():
        print(f"{entity:9} {name:12} {pattern}")

DATE      numeric      \b\d{1,2}[/.-]\d{1,2}[/.-](?:\d{4}|\d{2})\b
DATE      day_month    \b\d{1,2}(?:st|nd|rd|th)?\s+(?:January|February|March|April|May|June|July|August|September|October|November|December),?\s+\d{4}\b
DATE      month_day    \b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}\b
MONEY     kes          \b(?:KES|Kshs?|KSh)\.?\s?\d{1,3}(?:,\d{3})*(?:\.\d{2})?(?:/=)?
CASE_REF  case_number  \b(?:Civil|Criminal|Succession|Petition|Appeal|Application|Case|Cause|Suit)\b[A-Za-z .&]{0,40}?(?:No\.?\s*)?E?\d+\s+of\s+\d{4}\b
CASE_REF  neutral      \[\d{4}\]\s+(?:eKLR|[A-Z]+\s+\d+(?:\s+\(KLR\))?)


How each one works:

- **`numeric`**: 1–2 digits, a separator (`/`, `.` or `-`), 1–2 digits, another separator, then a 2- or 4-digit year. It doesn't check that the two separators match (`13/7-2022` would pass) or that the day and month are valid.
- **`day_month`**: a day with an optional `st/nd/rd/th` attached directly, a full month name, an optional comma, and a 4-digit year.
- **`month_day`**: the US-style order used in Kenya Law page metadata (`March 25, 2026`).
- **`kes`**: a currency marker (`KES`, `Ksh`, `Kshs`, `KSh`), an optional dot and space, digits with optional thousands commas and cents, and an optional `/=`.
- **`case_number`**: a keyword such as `Civil`, `Appeal` or `Case`, up to 40 characters of words, an optional `No.`, an optional `E` prefix (new-style case numbers), then `N of YYYY`. The match is non-greedy so it stops at the first number.
- **`neutral`**: a bracketed year, then `eKLR` (older citations) or a court code plus number, with an optional `(KLR)`.

All patterns are case-insensitive, which is why `NO.E107 OF 2022` and `Civil case No.` still match.

## 2. Run it on the cleaned slice

In [4]:
rows = []
for chunk in df.itertuples():
    for entity in extract(chunk.text):
        rows.append({"chunk_id": chunk.chunk_id, **entity})

preds = pd.DataFrame(rows)
preds.head()

,chunk_id,type,pattern,text,start,end
0,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,DATE,numeric,17/09/2007,632,642
1,https://new.kenyalaw.org/akn/ke/judgment/kemc/...,DATE,numeric,23/06/2022,270,280
2,https://new.kenyalaw.org/akn/ke/judgment/kehc/...,CASE_REF,neutral,[2017] KESC 2 (KLR),258,277
3,https://new.kenyalaw.org/akn/ke/judgment/kehc/...,CASE_REF,case_number,Petition E018 of 2023,649,670
4,https://new.kenyalaw.org/akn/ke/judgment/kehc/...,CASE_REF,neutral,[2024] KESC 34 (KLR),672,692


In [5]:
summary = preds.groupby(["type", "pattern"]).size().rename("matches").to_frame()
summary

matches
type     pattern             
CASE_REF case_number       46
         neutral          111
DATE     day_month         36
         month_day         14
         numeric           68
MONEY    kes               99

In [6]:
pd.DataFrame({
    "matches": preds.groupby("type").size(),
    "chunks with at least one": preds.groupby("type")["chunk_id"].nunique(),
})

,matches,chunks with at least one
type,,
CASE_REF,157,86
DATE,118,63
MONEY,99,52


In [7]:
print("chunks with no entity at all:", len(df) - preds["chunk_id"].nunique(), "of", len(df))

chunks with no entity at all: 228 of 381


## 3. What the matches look like

In [8]:
for name in preds["pattern"].unique():
    sample = preds.loc[preds["pattern"] == name, "text"].drop_duplicates().head(6).tolist()
    print(f"{name:12} {sample}")

numeric      ['17/09/2007', '23/06/2022', '13/7/2022', '29/4/2015', '26/3/2024', '1/2/2021']
neutral      ['[2017] KESC 2 (KLR)', '[2024] KESC 34 (KLR)', '[2016] eKLR', '[2018] eKLR', '[1974] EA 75', '[2024] KEHC 13993 (KLR)']
case_number  ['Petition E018 of 2023', 'Civil Suit 197 of 2018', 'Civil case No. E131 of 2024', 'Criminal Appeal No 53 of 2009', 'Appeal E012 of 2023', 'Case No. 6 of 2010']
day_month    ['8 November 2024', '1 March 2023', '12 July 2024', '3 June 2024', '4th December 2020', '6th December 2020']
kes          ['Ksh. 185,652/=', 'Ksh. 550/=', 'Kshs 4', 'Kshs. 7,000', 'Kshs.7,409', 'Kshs.7,779.45']
month_day    ['March 24, 2026', 'May 5, 2025', 'February 19, 2025', 'March 23, 2026', 'March 27, 2026', 'September 8, 2025']


To see the output in context, this marks each match inside the chunk it came from:

In [9]:
def annotate(text):
    out, last = "", 0
    for e in extract(text):
        if e["start"] < last:          # skip a match that overlaps the previous one
            continue
        out += text[last:e["start"]] + f"[{e['type']}: {e['text']}]"
        last = e["end"]
    return out + text[last:]

busiest = preds["chunk_id"].value_counts().index[0]
print(annotate(df.set_index("chunk_id").loc[busiest, "text"]))

QSG v MOG (Matrimonial [CASE_REF: Cause E024 of 2024]) [CASE_REF: [2025] KEKC 25 (KLR)] ([DATE: 19 February 2025]) (Judgment) Neutral citation: [CASE_REF: [2025] KEKC 25 (KLR)] Republic of Kenya In the Kadhis Court at Moyale Matrimonial [CASE_REF: Cause E024 of 2024] A Galgalo, PK [DATE: February 19, 2025] Between QSG Plaintiff and MOG Defendant Judgment 1. The Applicant filed a Notice of Motion dated 23 rd January 2025 seeking default judgment against the respondent who is defendant in the matrimonial [CASE_REF: cause E024 of 2024] in which she sought for custody of two minors under her care and protection, payment of dowry of 3 cows, maintenance at [MONEY: Kshs 30,000] p.m., shelter at [MONEY: Kshs 20,000] p.m., schools fees [MONEY: kshs 24,000] p.m., all to and maintenance debt at [MONEY: Kshs 25,000], cost of the matrimonial suit and for this application, and any further relief may lawfully be g


Even in the chunk with the most matches, two misses are visible. *"Notice of Motion dated 23 rd January 2025"* isn't tagged as a date because of the space in `23 rd`. *"Matrimonial Cause E024 of 2024"* is tagged only from `Cause` onwards, because `Matrimonial` isn't in the keyword list.

## 4. Known gaps, checked on purpose

The EDA and cleaning notebooks found formats this baseline doesn't handle. Rather than add patterns for them, they're listed here as expected misses so Phase 6 can check how often they happen in real text. Each line shows an input and what the baseline returns.

In [10]:
probes = [
    # should work
    "judgment delivered on 25th March, 2026",
    "an award of Ksh. 1,200,000/= in general damages",
    "in Civil Appeal No. E031 of 2023",
    "see Mwangi v Republic [2026] KEHC 3922 (KLR)",
    # known gaps
    "on 13 th August 2016",                    # spaced ordinal
    "THIS 23 RD DAY OF MARCH, 2026",           # formal delivery line
    "Kenya Shillings One Million",             # amount in words
    "the purported Kshs 4, 000 recovered",     # stray space inside the number
    "a fine of Sh. 50,000",                    # 'Sh.' not in the currency list
    "Environment & Land Case E018 of 2025",    # court name not in the keyword list
    "under the Land Act No. 6 of 2012",        # a statute, not a case
]

for p in probes:
    print(f"{p:42} -> {[e['text'] for e in extract(p)]}")

judgment delivered on 25th March, 2026     -> ['25th March, 2026']
an award of Ksh. 1,200,000/= in general damages -> ['Ksh. 1,200,000/=']
in Civil Appeal No. E031 of 2023           -> ['Civil Appeal No. E031 of 2023']
see Mwangi v Republic [2026] KEHC 3922 (KLR) -> ['[2026] KEHC 3922 (KLR)']
on 13 th August 2016                       -> []
THIS 23 RD DAY OF MARCH, 2026              -> []
Kenya Shillings One Million                -> []
the purported Kshs 4, 000 recovered        -> ['Kshs 4']
a fine of Sh. 50,000                       -> []
Environment & Land Case E018 of 2025       -> ['Case E018 of 2025']
under the Land Act No. 6 of 2012           -> []


What the probes show:
- **Spaced ordinals and formal delivery lines** (`13 th August`, `23 RD DAY OF MARCH`) give no date.
- **Amounts in words** give nothing, and **`Kshs 4, 000`** is cut to `Kshs 4`, the wrong amount.
- **`Sh.`** isn't recognised as a currency marker.
- **`Environment & Land Case E018 of 2025`** only matches from `Case` onwards, so the extracted reference is incomplete.
- **Statute references** such as `Land Act No. 6 of 2012` don't match, because none of the case keywords is present. That's only luck: `Petition No. 6 of 2012` would match whether it was a case or not.

These are hypotheses until they're measured against hand labels in Phase 6.

## Summary

- Three entity types, six patterns, about 35 lines of code in `src/baseline.py`.
- It finds entities in a minority of chunks, as the EDA frequency counts suggested.
- It's fast and easy to explain, but brittle: it matches the formats written into it and nothing else, and it knows nothing about context (a date of birth and a judgment date look the same to it).
- The probes above list the expected failure modes; Phase 6 measures precision, recall and F1 against hand labels.